## GCS Command Forwarding — Sanity Check

Single drone flies a straight AUTO mission heading north.
Once the first real waypoint is active (`MISSION_CURRENT.seq >= 1`), the GCS
switches it to GUIDED mode and sends it east via `DO_REPOSITION`.

If the drone visibly changes direction, the GCS→Logic→SITL command path works.

In [ ]:
from simulator import Oracle, Simulator
from simulator.config import PARAMS_PATH, Color, Model
from simulator.entities import SimGCS, SimVehicle
from simulator.helpers import clean
from simulator.helpers.coordinates import ENU, ENUPose, GRAPose
from simulator.planner import AutoPlan
from simulator.visualizer import Gazebo, GazMarker

clean()


## Origin and waypoints

In [ ]:
gra_origin = GRAPose(lat=-35.3633280, lon=149.1652241, alt=0, heading=0)
enu_origin = ENUPose(x=0, y=0, z=gra_origin.alt, heading=gra_origin.heading)

home = ENUPose(0, 0, 0, 0)
speed = 5.0  # m/s
cruise_alt = 10.0  # m
model = Model.IRIS
sysid = 1

# Mission seq: seq=0 home, seq=1 TAKEOFF, seq=2 flying to north_100,
#              seq=3 flying to north_200 ← intervention fires here
home_wp = ENU(x=0, y=0, z=0)
north_100 = ENU(x=0, y=100, z=cruise_alt)
north_200 = ENU(x=0, y=200, z=cruise_alt)
mission_wps = [home_wp, north_100, north_200]

# Intervention target: 100 m east of origin at cruise altitude
east_target = gra_origin.unpose().to_abs(ENU(x=100, y=0, z=cruise_alt))
print(f"Intervention target: lat={east_target.lat:.7f}, lon={east_target.lon:.7f}")


## Vehicle

In [ ]:
mission_path = "simulator/planner/missions/sanity_north.waypoints"

plan = AutoPlan.from_relative_path(
    name="north_mission",
    sysid=sysid,
    gra_origin=gra_origin,
    relative_home=home,
    relative_path=mission_wps,
    mission_path=mission_path,
    navigation_speed=speed,
    firmware=model.firmware,
)

vehicle = SimVehicle.from_relative(
    sysid=sysid,
    gcss=[SimGCS(name=f"{Color.BLUE.name}_{Color.BLUE.emoji}")],
    plan=plan,
    color=Color.BLUE,
    enu_origin=enu_origin,
    relative_home=home,
    relative_path=mission_wps,
    model=model,
)


## Visualizer

In [ ]:
gaz = Gazebo(gra_origin, world_path="simulator/visualizer/gazebo/worlds/runway.world")
origin_marker = GazMarker(
    name="origin",
    group="origin",
    pos=enu_origin.unpose(),
    color=Color.WHITE,
)
gaz.markers.append(origin_marker)


## Simulator

In [ ]:
orac = Oracle()

orac.add_vehicle(vehicle)

# Trigger at seq=3: drone has reached north_100 and is flying to north_200.
# `orac.intervention` is keyed by sysid, so this targets only this vehicle.
orac.intervention[sysid] = {
    "trigger_seq": 3,
    "target_lat": east_target.lat,
    "target_lon": east_target.lon,
    "target_alt": cruise_alt,
}

simulator = Simulator(oracle=orac, visualizer=gaz, verbose=1)

simulator.preview()


In [ ]:
simulator.run()
